[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EddiazOS/LC_data_review/blob/main/05_Analisis_Reproducible/notebooks/01_nivel_1_ocurrencia_y_descriptiva.ipynb)

# Notebook 01: Nivel 1 — Ocurrencia y Estadística Descriptiva
### Subsección 3.1 del Manuscrito: Frecuencias de Detección, Niveles de Concentración y Tabla 1
**Proyecto LC-GC / Indoor Dust — Universidad de Cartagena & Universidad de Amberes**


In [ ]:
# Celda 1: Configuración de Entorno (Google Colab / Local) y Pre-requisitos
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Entorno Google Colab detectado.")
    if not os.path.exists('LC_data_review'):
        !git clone https://github.com/EddiazOS/LC_data_review.git
    if os.path.exists('LC_data_review/05_Analisis_Reproducible/datos_curados'):
        DATA_DIR = 'LC_data_review/05_Analisis_Reproducible/datos_curados'
        OUT_TABLES = 'LC_data_review/05_Analisis_Reproducible/outputs/tablas'
        OUT_FIGS = 'LC_data_review/05_Analisis_Reproducible/outputs/figuras'
    else:
        DATA_DIR = '../datos_curados'
        OUT_TABLES = '../outputs/tablas'
        OUT_FIGS = '../outputs/figuras'
else:
    print("💻 Entorno Local detectado.")
    DATA_DIR = '../datos_curados' if os.path.exists('../datos_curados') else '05_Analisis_Reproducible/datos_curados'
    OUT_TABLES = '../outputs/tablas' if os.path.exists('../outputs/tablas') else '05_Analisis_Reproducible/outputs/tablas'
    OUT_FIGS = '../outputs/figuras' if os.path.exists('../outputs/figuras') else '05_Analisis_Reproducible/outputs/figuras'

os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)
print(f"Directorio de datos activo: {DATA_DIR}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración gráfica para publicación científica
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
print("Librerías y entorno gráfico listos.")


In [ ]:
# Celda 2: Auto-Carga de Datos Curados y Preparación de Métodos de Censura
df_muestras = pd.read_csv(os.path.join(DATA_DIR, 'muestras_metadata.csv'))
df_compuestos = pd.read_csv(os.path.join(DATA_DIR, 'compuestos_metadata.csv'))
df_conc = pd.read_csv(os.path.join(DATA_DIR, 'concentraciones_ng_g.csv'))
df_det = pd.read_csv(os.path.join(DATA_DIR, 'estado_deteccion.csv'))
df_loqs = pd.read_csv(os.path.join(DATA_DIR, 'limites_cuantificacion.csv'))

fam_map = dict(zip(df_compuestos['acronym'], df_compuestos['family']))
acronyms = list(df_compuestos['acronym'])

# Método A: Imputación por LOQ (enfoque de la tesis)
df_conc_A = df_conc.copy()

# Método B: Imputación por LOQ / 2 (estándar EPA)
df_conc_B = df_conc.copy()
for col in acronyms:
    loq_val = df_loqs.loc[df_loqs['acronym'] == col, 'loq_imputed_sample_ng_g'].values[0]
    mask_nd = (df_det[col] == 0)
    df_conc_B.loc[mask_nd, col] = loq_val / 2.0

print(f"Matrices preparadas exitosamente:")
print(f" - Muestras: {len(df_muestras)} ({df_muestras['microenvironment'].value_counts().to_dict()})")
print(f" - Analitos: {len(acronyms)}")


In [ ]:
# Celda 3: Cálculo de Frecuencias de Detección (DF %) Global y por Microambiente
df_res_df = []
for col in acronyms:
    fam = fam_map[col]
    det_total = df_det[col].sum()
    pct_total = (det_total / 44.0) * 100.0

    det_car = df_det.loc[df_muestras['microenvironment'] == 'Car', col].sum()
    pct_car = (det_car / 12.0) * 100.0

    det_house = df_det.loc[df_muestras['microenvironment'] == 'House', col].sum()
    pct_house = (det_house / 24.0) * 100.0

    det_office = df_det.loc[df_muestras['microenvironment'] == 'Office', col].sum()
    pct_office = (det_office / 8.0) * 100.0

    df_res_df.append({
        'Family': fam,
        'Congener': col,
        'DF_Total_pct': pct_total,
        'Det_Total_n': det_total,
        'DF_Car_pct': pct_car,
        'DF_House_pct': pct_house,
        'DF_Office_pct': pct_office
    })

df_df_summary = pd.DataFrame(df_res_df)
df_df_summary.to_csv(os.path.join(OUT_TABLES, 'frecuencias_deteccion_resumen.csv'), index=False)
display(df_df_summary.sort_values(by='DF_Total_pct', ascending=False).head(15))


In [ ]:
# Celda 4: Comparación de Estadística Descriptiva (Método A vs Método B)
def compute_stats(df_data, df_meta):
    records = []
    for comp in acronyms:
        fam = fam_map[comp]
        for micro in ['Car', 'House', 'Office', 'Overall']:
            if micro == 'Overall':
                vals = df_data[comp].values
            else:
                idxs = df_meta[df_meta['microenvironment'] == micro].index
                vals = df_data.loc[idxs, comp].values

            records.append({
                'Family': fam,
                'Congener': comp,
                'Microenvironment': micro,
                'N': len(vals),
                'Mean': np.mean(vals),
                'SD': np.std(vals, ddof=1),
                'Median': np.median(vals),
                'Q1': np.percentile(vals, 25),
                'Q3': np.percentile(vals, 75),
                'IQR': np.percentile(vals, 75) - np.percentile(vals, 25),
                'Min': np.min(vals),
                'Max': np.max(vals)
            })
    return pd.DataFrame(records)

df_stats_A = compute_stats(df_conc_A, df_muestras)
df_stats_B = compute_stats(df_conc_B, df_muestras)

df_comp_med = pd.merge(
    df_stats_A[['Family', 'Congener', 'Microenvironment', 'Median']],
    df_stats_B[['Family', 'Congener', 'Microenvironment', 'Median']],
    on=['Family', 'Congener', 'Microenvironment'],
    suffixes=('_Metodo_A_LOQ', '_Metodo_B_LOQ_div_2')
)
df_comp_med['Diff_pct'] = ((df_comp_med['Median_Metodo_A_LOQ'] - df_comp_med['Median_Metodo_B_LOQ_div_2']) / df_comp_med['Median_Metodo_A_LOQ']) * 100

print("Comparación de medianas para analitos con censura (< LOQ) en Casas:")
display(df_comp_med[(df_comp_med['Microenvironment'] == 'House') & (df_comp_med['Diff_pct'] > 0)].sort_values(by='Diff_pct', ascending=False))


In [ ]:
# Celda 5: Balance de Masa Acumulado Muestra a Muestra
for name, df_d in [('Metodo_A', df_conc_A), ('Metodo_B', df_conc_B)]:
    df_m = df_muestras.copy()

    aps_cols = [c for c in acronyms if fam_map[c] == 'APs']
    pfrs_cols = [c for c in acronyms if fam_map[c] == 'PFRs']
    phs_cols = [c for c in acronyms if fam_map[c] == 'PHs']

    df_m['Sum_APs'] = df_d[aps_cols].sum(axis=1)
    df_m['Sum_PFRs'] = df_d[pfrs_cols].sum(axis=1)
    df_m['Sum_PHs'] = df_d[phs_cols].sum(axis=1)
    df_m['Sum_Total'] = df_m['Sum_APs'] + df_m['Sum_PFRs'] + df_m['Sum_PHs']

    print(f"=== BALANCE DE MASA ACUMULADO ({name}) ===")
    print(f"Sum_Total: Mediana = {df_m['Sum_Total'].median():,.1f} ng/g | Media = {df_m['Sum_Total'].mean():,.1f} +/- {df_m['Sum_Total'].std():,.1f} ng/g")
    print(f"Sum_PHs  : Mediana = {df_m['Sum_PHs'].median():,.1f} ng/g ({df_m['Sum_PHs'].sum()/df_m['Sum_Total'].sum()*100:.1f}%)")
    print(f"Sum_APs  : Mediana = {df_m['Sum_APs'].median():,.1f} ng/g ({df_m['Sum_APs'].sum()/df_m['Sum_Total'].sum()*100:.1f}%)")
    print(f"Sum_PFRs : Mediana = {df_m['Sum_PFRs'].median():,.1f} ng/g ({df_m['Sum_PFRs'].sum()/df_m['Sum_Total'].sum()*100:.1f}%)\n")


In [ ]:
# Celda 6: Generación de la Figura 1 (Distribución log10 por Microambiente)
fig, ax = plt.subplots(figsize=(8, 5.5))

df_plot = df_muestras.copy()
df_plot['Sum_Total'] = df_conc_A[[c for c in acronyms]].sum(axis=1)
df_plot['log10_Total'] = np.log10(df_plot['Sum_Total'])

palette = {'Car': '#E67E22', 'House': '#27AE60', 'Office': '#2980B9'}
order = ['Car', 'House', 'Office']

sns.boxplot(data=df_plot, x='microenvironment', y='log10_Total', order=order, palette=palette, width=0.45, ax=ax, boxprops=dict(alpha=0.8))
sns.stripplot(data=df_plot, x='microenvironment', y='log10_Total', order=order, color='black', alpha=0.6, jitter=0.2, size=7, ax=ax)

ax.set_xlabel('Microenvironment', fontweight='bold', labelpad=8)
ax.set_ylabel('Total Additives (log10 ng/g)', fontweight='bold', labelpad=8)
ax.set_xticklabels(['Vehicles\n(n = 12)', 'Homes\n(n = 24)', 'Offices\n(n = 8)'])
ax.grid(axis='y', linestyle='--', alpha=0.4)

fig_path = os.path.join(OUT_FIGS, 'figura_1_distribucion_log10.png')
plt.savefig(fig_path)
plt.show()
print(f"Figura 1 guardada en: {fig_path}")
